### Libraries

On runAI, we use the following libraries on top of the pytorch image

In [ ]:
%%capture
!pip install tqdm pandas numpy scikit-learn nltk bertopic sentence-transformers cuml accelerate

In [1]:
import os
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
import nltk
from nltk.corpus import stopwords

from sentence_transformers import SentenceTransformer
import torch
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer

try:
    from cuml.manifold import UMAP
    from cuml.cluster import HDBSCAN
except ImportError:
    from umap import UMAP
    from hdbscan import HDBSCAN

/home/matteo/Documents/Projects/DemocraSci-AM/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tqdm.pandas()

# Speech Data
They are fetched using the `paragraph_extraction.py` script which matches the Specch_ID with the corresponding metadata in the Neo4J database.

In [4]:
df = pd.read_parquet("data/clean_paragraphs.parquet")
df

,Speech_ID,Paragraph_ID,TEXT_SPAN,role,language_of_text,name
0,81893,81893_p0,La commission propose de ne pas donner suite à...,President,FR,Bugnon André
1,81894,81894_p0,"Ich möchte Sie nur nochmals bitten, das Result...",For the committee,DE,Widmer Hans
2,81894,81894_p1,"In diesem Sinne bitte ich Sie nochmals, den An...",For the committee,DE,Widmer Hans
3,81895,81895_p0,Der Bundesrat beantragt die Annahme des Postul...,Member of the federal council,DE,Schmid Samuel
4,81895,81895_p1,Aber es geht hier noch um andere Fragen. Es ge...,Member of the federal council,DE,Schmid Samuel
...,...,...,...,...,...,...
303379,288681,288681_p11,"En commission, une minorité avait, quant à ell...",For the committee,FR,Python Valentine
303380,288681,288681_p12,La proposition individuelle Bäumle déposée hie...,For the committee,FR,Python Valentine
303381,288681,288681_p13,En ce qui concerne le cas précis de cette prop...,For the committee,FR,Python Valentine
303382,288681,288681_p14,La majorité de la commission désire prendre en...,For the committee,FR,Python Valentine


The bills topics are instead manually defined here and come from the SwissDomainBERT2 model.

In [ ]:
bills_topics = ['Civil Rights', 'Technology, Communication', 'International Affairs', 'Government Operations', 'Economy', 'Education, Research', 'Culture, Arts, Religion, Sport, Entertainment', 'Public Finances, Taxation', 'Agriculture', 'Social Welfare', 'Labor', 'Health', 'National Security, Armed Forces, Defense', 'Environment', 'Energy', '(Public) Transportation', 'Finances, Financial Sector', 'Spatial Planing, Civil Engineering', 'Migration', 'Parliamentary Procedure', 'Criminal Law']
print(f"Found {len(bills_topics)} unique topics: {bills_topics}")

# Embeddings (Qwen3-0.6B)

Instantiate the Embedding model

In [ ]:
# Load the embedding model
EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-0.6B"
embedder = SentenceTransformer(
    EMBEDDING_MODEL,
    trust_remote_code=True,
    model_kwargs={
        "torch_dtype": torch.bfloat16,
        "attn_implementation": "sdpa",
    },
    processor_kwargs={"padding_side": "left"},
)

Uncomment to compute again the Qwen embeddings over the paragraphs.

In [ ]:
# embeddings = embedder.encode(
#    texts,
#    normalize_embeddings=True,
#    batch_size=64,
#    show_progress_bar=True,
# )

We store the precomputed embeddings to avoid recomputing them every time.

In [ ]:
# np.save(f"paragraph_embeddings_{EMBEDDING_MODEL.split("/")[-1]}.npy", embeddings)

Load back pretrained embeddings and normalize to unit length for cosine similarity search.

In [ ]:
# Load pre-computed embeddings
embeddings = np.load(f"paragraph_embeddings_{EMBEDDING_MODEL.split("/")[-1]}.npy")

# Normalize embeddings to unit length
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
embeddings.shape

# BERTopic Model (UMAP + HDBSCAN)

Core part of the code, we need to define several hyperparameters for the dimensionality reduction, clustering and zero-shot assingment of the 21 topics (by means of minimum cosine similarity).

The `ZEROSHOT_MIN_SIMILARITY` is a hyperparameter that can be tuned to assign in a zero-shot fashion one of the 21 Bills Topics to the extracted speech paragraphs.

For UMAP, we need to define the number of dimensions to reduce to, the number of neighbors to consider and the distance metric.

For HDBSCAN, we have several hyperparameters to tune, among them the minimum cluster size, the minimum number of samples in a cluster, the cluster selection method (EOM or Leaf - depending whether we want to start from a bigger cluster and then split it or to start from smaller clusters and then merge them) and the cluster selection epsilon (which is a distance threshold to split clusters that are too far apart).

There are some rules of thumb to tune these hyperparameters, but in general it is a good idea to try different combinations and see which one gives the best results in terms of coherence and interpretability of the topics.

>The more interpretable the topics are, the better the LLM refinement will be, which is crucial for the downstream tasks of stance detection.

In [ ]:
# Hyperparameters for BERTopic
RANDOM_SEED = 42

# UMAP
N_NEIGHBORS = 20
N_COMPONENTS = 5
METRIC = "cosine"

# HDBSCAN
MIN_CLUSTER_SIZE = 150
MIN_SAMPLES = 10
CLUSTER_SELECTION_METHOD = "leaf"  # "eom" (excess of mass) or "leaf"
CLUSTER_SELECTION_EPSILON = 0.1

# Zero-Shot topic
ZEROSHOT_MIN_SIMILARITY = 0.65  # Min cos similarity between speech embedding and predefined topic embedding to assign the topic to the speech

We use NLTK to load common stopwords in German, French and Italian to remove them from the extracted topics, which helps the CountVectorizer to better capture the important words in the topics and to improve the coherence of the topics.

The cTF-IDF is a variant of the TF-IDF that is computed for multiple classes by joining all documents per class. Each class is converted to a single document instead of set of documents. The frequency of each word x is extracted for each class c and is l1 normalized. This constitutes the term frequency. Then, the term frequency is multiplied with IDF which is the logarithm of 1 plus the average number of words per class A divided by the frequency of word x across all classes.

TF-IDF is used to extract the most important words for each topic, which are then used to "represent" the cluster by means of these keywords.

In [ ]:
# Download stopwords for German, French, and Italian
nltk.download("stopwords")

# Combine stopwords from German, French, and Italian
stop_words_multi = (
    stopwords.words("german") + stopwords.words("french") + stopwords.words("italian")
)

# UMAP for dimensionality reduction
umap_model = UMAP(
    n_neighbors=N_NEIGHBORS,
    n_components=N_COMPONENTS,
    metric=METRIC,
    random_state=RANDOM_SEED,
)

# HDBSCAN for clustering
hdbscan_model = HDBSCAN(
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=MIN_SAMPLES,
    cluster_selection_method=CLUSTER_SELECTION_METHOD,
    cluster_selection_epsilon=CLUSTER_SELECTION_EPSILON,
    prediction_data=True,
)

# CountVectorizer with custom stop words (German, French, Italian)
vectorizer_model = CountVectorizer(
    stop_words=stop_words_multi,
)
# TF-IDF transformer that reduces the importance of frequent words across topics
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

# Initialize BERTopic with the specified models and parameters
topic_model = BERTopic(
    embedding_model=embedder,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    verbose=True,
    zeroshot_topic_list=bills_topics,
    zeroshot_min_similarity=ZEROSHOT_MIN_SIMILARITY,
)

If we have already saved the model locally, we can load it back otherswise we fit the model and save it for future use.

In [ ]:
overwrite_model = False
if os.path.exists(f"bertopic_model_{len(texts)}_samples") and not overwrite_model:
    print("Model found, loading...")
    topic_model = BERTopic.load(
        f"bertopic_model_{len(texts)}_samples",
        embedding_model=embedder,
    )
else:
    print("No model found, fitting a new one...")
    topics, probs = topic_model.fit_transform(texts, embeddings=embeddings)

We inspect the resulting topics

In [ ]:
topic_model.get_topic_info()

Now, `reduce_outliers` method basically tries to shrink the -1 cluster of DBSCAN (the outliers) by reassigning them to the closest cluster (given some heuristic).

These can be:
- calculating the c-TF-IDF representations of outlier documents and assigning them to the best matching c-TF-IDF representations of non-outlier topics.
- Using the topic-document probabilities to assign topics
- Using the topic-document distributions to assign topics
- Using the embeddings of each outlier documents, find the best matching topic embedding using cosine similarity.

Whether we should reduce the outliers or not highly depends on the amount of outliers that we have. If they are too many then either most of the paragraphs are just procedural/generic speeches that are not really relevant for the downstream task of stance detection, or we need to tune the hyperparameters of UMAP and HDBSCAN to get more coherent topics and less outliers.

I did not experiment too much with this reduction, but it is worth trying to see if it can improve the results of the downstream task. The one I used is the one based on the embeddings, which is the most intuitive and it is based on the idea that if a paragraph is an outlier it is probably because it is not really similar to any of the topics, but maybe it is still similar to some of them and we can assign it to the closest one.

In [ ]:
new_topics = topic_model.reduce_outliers(
    texts, topics, strategy="embeddings", embeddings=embeddings
)
topic_model.update_topics(texts, topics=new_topics)

In [ ]:
topic_model.get_topic_info()

In [ ]:
topic_model.visualize_topics()

In [ ]:
top_n = 20
topic_model.visualize_heatmap(top_n_topics=top_n, title=f"Top {top_n} Topics Heatmap")

In [ ]:
topics_per_class = topic_model.topics_per_class(texts, df["name"])

In [ ]:
topic_model.visualize_topics_per_class(topics_per_class)

# Topic Representation Refinement (via SDSC Qwen)

Here you will need to pass your API KEY for the SDSC Qwen.

In [ ]:
import json

with open("/home/matteo/.config/io.datasette.llm/keys.json", "r") as f:
    keys = json.load(f)

API_KEY = keys["sdsc-key"]

And the following function conveniently wraps the API call with cURL to the vLLM endpoint where the Qwen is deployed.

Several parameters can be tuned, but more importantly, you can change the prompt and thinking process to get better topic representations.

In [ ]:
import requests
from const import DEFAULT_SYSTEM_PROMPT, DEFAULT_CHAT_PROMPT


def extract_topic_label(
    system_prompt: str = DEFAULT_SYSTEM_PROMPT,
    chat_prompt: str = DEFAULT_CHAT_PROMPT,
    temperature: float = 0.7,
    top_p: float = 0.8,
    top_k: int = 20,
    max_tokens: int = 256,
    presence_penalty: float = 1.5,
    enable_thinking: bool = False,
    endpoint: str = "https://vllm-gateway-runai-sharedllm-ralf.inference.compute.datascience.ch/v1/chat/completions",
    model: str = "Qwen/Qwen3.6-35B-A3B-FP8",
):
    r = requests.post(
        endpoint,
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {API_KEY}",
        },
        json={
            "model": model,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": chat_prompt},
            ],
            "temperature": temperature,
            "top_p": top_p,
            "top_k": top_k,
            "max_tokens": max_tokens,
            "presence_penalty": presence_penalty,
            "chat_template_kwargs": {"enable_thinking": enable_thinking},
        },
    )

    if r.ok:
        response = r.json()
        clean_response = response["choices"][0]["message"]["content"].strip()
        topic = clean_response.replace("topic:", "").strip()
        return topic
    else:
        print(f"Error: {r.status_code} - {r.text}")
        return None

This is the wrapper around the BaseRepresentation class of BERTopic to be able to query Qwen directly when refining the topic representations inside the BERTopic pipeline.

In [ ]:
from bertopic.representation._base import BaseRepresentation


class QwenTopic(BaseRepresentation):
    def __init__(
        self,
        bills_topics: list[str] = bills_topics,
        prompt=DEFAULT_CHAT_PROMPT,
        system_prompt=DEFAULT_SYSTEM_PROMPT,
        nr_docs: int = 10,
        nr_samples: int = 1000,
        temperature: float = 0.7,
        top_p: float = 0.8,
        top_k: int = 20,
        max_tokens: int = 256,
        presence_penalty: float = 1.5,
        enable_thinking: bool = False,
        diversity: float | None = None,
    ):
        self.bills_topics = bills_topics
        self.prompt = prompt
        self.system_prompt = system_prompt
        self.nr_docs = nr_docs
        self.nr_samples = nr_samples
        self.temperature = temperature
        self.top_p = top_p
        self.top_k = top_k
        self.max_tokens = max_tokens
        self.presence_penalty = presence_penalty
        self.enable_thinking = enable_thinking
        self.diversity = diversity

        self.prompts_ = []

    def extract_topics(self, topic_model, documents, c_tf_idf, topics):
        repr_docs_mappings, _, _, _ = topic_model._extract_representative_docs(
            c_tf_idf, documents, topics, self.nr_samples, self.nr_docs, self.diversity
        )

        updated_topics = {}
        for topic, docs in tqdm(
            repr_docs_mappings.items(), desc="Extracting topics with Qwen"
        ):
            prompt = self._create_prompt(docs, topic, topics)
            self.prompts_.append(prompt)

            label = extract_topic_label(
                chat_prompt=prompt,
                max_tokens=self.max_tokens,
                enable_thinking=self.enable_thinking,
            )
            updated_topics[topic] = [(label, 1)]

        return updated_topics

    def _create_prompt(self, docs, topic, topics):
        keywords = next(zip(*topics[topic]))

        prompt = self.prompt
        if "[BILLS_TOPICS]" in prompt:
            prompt = prompt.replace("[BILLS_TOPICS]", ", ".join(self.bills_topics))
        if "[KEYWORDS]" in prompt:
            prompt = prompt.replace("[KEYWORDS]", ", ".join(keywords))
        if "[DOCUMENTS]" in prompt:
            prompt = self._replace_documents(prompt, docs)

        return prompt

    @staticmethod
    def _replace_documents(prompt, docs):
        to_replace = ""
        for doc in docs:
            to_replace += f"- {doc}\n"
        prompt = prompt.replace("[DOCUMENTS]", to_replace)
        return prompt

    def get_prompts(self):
        """Returns the prompts used for topic extraction."""
        return self.prompts_

In [ ]:
# Update topics with the new representation model based on Qwen
topic_model.update_topics(
    texts,
    representation_model=QwenTopic(
        nr_docs=10,
        nr_samples=1000,
        temperature=0.0,
        max_tokens=512,
        enable_thinking=False,
    ),
)

In [ ]:
topic_model.get_topic_info()

In [ ]:
topic_model.save(
    path=f"bertopic_model_{len(texts)}_samples_qwen_refined",
    serialization="safetensors",
    save_embedding_model=True,
    save_ctfidf=True,
)

At the end we save the document topics, map back to the original dataframe and save the resulting dataframe with the new topic assignments.

In [ ]:
doc_topics = topic_model.get_document_info(texts)
doc_topics.to_csv("bertopic_document_topics.csv", index=False)
doc_topics

In [ ]:
doc_topics["Representation"].value_counts(ascending=True).plot(
    kind="barh",
    title="Distribution of topics",
    figsize=(10, 25),
    xlabel="Number of paragraphs",
    ylabel="Topics",
)

In [ ]:
full_df = pd.concat([df, doc_topics], axis=1, join="inner")
full_df = full_df.drop(columns=["Document"])
full_df.to_json(
    "speeches_with_topics.jsonl", orient="records", lines=True, force_ascii=False
)